In [ ]:
"""
Task 3: Energy Consumption Time Series Forecasting
===================================================
Dataset: Household Power Consumption (synthetic, matching UCI schema)
Models:  ARIMA (manual), Moving Average + Linear Trend, XGBoost-style (GBR from sklearn)
Metrics: MAE, RMSE
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ─────────────────────────────────────────────
# 1. GENERATE REALISTIC POWER CONSUMPTION DATA
# ─────────────────────────────────────────────
def generate_power_data(start='2007-01-01', periods_days=60):
    """Generate hourly household power consumption (kW) with realistic patterns."""
    n_hours = periods_days * 24
    idx     = pd.date_range(start=start, periods=n_hours, freq='h')

    # Base hourly profile (0-23)
    hourly_pattern = np.array([
        0.3, 0.25, 0.22, 0.20, 0.20, 0.25,   # 0-5: night
        0.35, 0.60, 0.90, 0.80, 0.70, 0.65,   # 6-11: morning
        0.70, 0.65, 0.60, 0.65, 0.75, 0.95,   # 12-17: afternoon
        1.20, 1.35, 1.30, 1.10, 0.80, 0.50    # 18-23: evening peak
    ])

    hours    = idx.hour.values
    weekdays = idx.dayofweek.values
    months   = idx.month.values

    # Seasonal component (winter peak)
    seasonal = 1.0 + 0.3 * np.cos(2 * np.pi * (months - 1) / 12)

    # Weekend effect
    weekend_mult = np.where(weekdays >= 5, 1.15, 1.0)

    # Base signal
    base = hourly_pattern[hours] * seasonal * weekend_mult

    # Add trend
    trend = 0.0005 * np.arange(n_hours)

    # Add noise
    noise = np.random.normal(0, 0.06, n_hours)
    occasional_spike = np.random.choice([0, 0.4], n_hours, p=[0.97, 0.03])

    power = base + trend + noise + occasional_spike
    power = np.clip(power, 0.1, 3.5)

    return pd.Series(power, index=idx, name='Global_active_power')

ts = generate_power_data(periods_days=60)
print(f"Time series shape: {ts.shape}")
print(f"Date range: {ts.index[0]} → {ts.index[-1]}")
print(f"Stats: mean={ts.mean():.3f}  std={ts.std():.3f}  min={ts.min():.3f}  max={ts.max():.3f}")

# Resample to 4-hourly for tractability
ts_4h  = ts.resample('4h').mean()
print(f"\nResampled (4h) shape: {ts_4h.shape}")

# ─────────────────────────────────────────────
# 2. TRAIN / TEST SPLIT  (last 7 days = test)
# ─────────────────────────────────────────────
n_test   = 7 * 6   # 7 days × 6 periods/day
ts_train = ts_4h.iloc[:-n_test]
ts_test  = ts_4h.iloc[-n_test:]
print(f"Train: {len(ts_train)}  |  Test: {len(ts_test)}")

# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING (for ML model)
# ─────────────────────────────────────────────
def make_features(series, lags=12):
    df = pd.DataFrame({'y': series})
    df['hour']     = df.index.hour
    df['dayofweek']= df.index.dayofweek
    df['month']    = df.index.month
    df['is_weekend']= (df['dayofweek'] >= 5).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin']  = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']  = np.cos(2 * np.pi * df['dayofweek'] / 7)
    for lag in range(1, lags + 1):
        df[f'lag_{lag}'] = df['y'].shift(lag)
    df['rolling_mean_6']  = df['y'].shift(1).rolling(6).mean()
    df['rolling_std_6']   = df['y'].shift(1).rolling(6).std()
    df['rolling_mean_12'] = df['y'].shift(1).rolling(12).mean()
    return df.dropna()

full_feat = make_features(ts_4h)
feat_cols = [c for c in full_feat.columns if c != 'y']

# Split ensuring index alignment
train_feat = full_feat[full_feat.index <= ts_train.index[-1]]
test_feat  = full_feat[full_feat.index.isin(ts_test.index)]

X_tr = train_feat[feat_cols]
y_tr = train_feat['y']
X_te = test_feat[feat_cols]
y_te = test_feat['y']

print(f"\nFeature matrix: {X_tr.shape[1]} features")

# ─────────────────────────────────────────────
# 4. ADF TEST
# ─────────────────────────────────────────────
def adf_simple(series):
    """Simple ADF-like check using variance ratio."""
    x   = series.values
    var_full = np.var(x)
    var_diff = np.var(np.diff(x))
    ratio    = var_diff / var_full
    return ratio  # ratio < 1 suggests stationarity

ratio = adf_simple(ts_train)
print(f"\nVariance ratio (diff/orig): {ratio:.4f}")
print("  → " + ("Likely Stationary " if ratio < 1.5 else "Possible Non-Stationarity "))

# ─────────────────────────────────────────────
# 5. MODEL 1: ARIMA (manual AR(p) with differencing)
# ─────────────────────────────────────────────
print("\nFitting ARIMA-style model (AR lag regression with differencing)...")

def fit_arima_manual(train, test, p=12, d=1):
    """
    Manual ARIMA(p,d,0) using OLS regression on lagged differenced series.
    """
    # Differencing
    if d == 1:
        y_diff = np.diff(train.values)
    else:
        y_diff = train.values.copy()

    # Build lag matrix
    X_lag, y_lag = [], []
    for i in range(p, len(y_diff)):
        X_lag.append(y_diff[i-p:i])
        y_lag.append(y_diff[i])
    X_lag = np.array(X_lag)
    y_lag = np.array(y_lag)

    # Fit OLS
    model = Ridge(alpha=1.0)
    model.fit(X_lag, y_lag)

    # Forecast (recursive)
    last_diff = list(y_diff[-p:])
    last_val  = train.values[-1]
    preds     = []
    for _ in range(len(test)):
        x_in   = np.array(last_diff[-p:]).reshape(1, -1)
        d_pred = model.predict(x_in)[0]
        val    = last_val + d_pred
        preds.append(val)
        last_diff.append(d_pred)
        last_val = val
    return np.array(preds)

arima_pred = fit_arima_manual(ts_train, ts_test, p=12, d=1)
arima_mae  = mean_absolute_error(ts_test.values, arima_pred)
arima_rmse = np.sqrt(mean_squared_error(ts_test.values, arima_pred))
print(f"  ARIMA(12,1,0)  MAE={arima_mae:.4f}  RMSE={arima_rmse:.4f}")

# ─────────────────────────────────────────────
# 6. MODEL 2: GBR (XGBoost-style)
# ─────────────────────────────────────────────
print("Fitting GradientBoostingRegressor (XGBoost-style)...")
gbr = GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                learning_rate=0.05, subsample=0.8,
                                random_state=42)
gbr.fit(X_tr, y_tr)
gbr_pred = gbr.predict(X_te)
gbr_mae  = mean_absolute_error(y_te, gbr_pred)
gbr_rmse = np.sqrt(mean_squared_error(y_te, gbr_pred))
print(f"  GBR     MAE={gbr_mae:.4f}  RMSE={gbr_rmse:.4f}")

# ─────────────────────────────────────────────
# 7. MODEL 3: PROPHET-LIKE (Fourier + Linear Trend)
# ─────────────────────────────────────────────
print("Fitting Prophet-style (Fourier seasonality) model...")

def make_fourier_features(index, n_terms=3):
    t = np.arange(len(index)) / len(index)
    cols = {}
    for k in range(1, n_terms + 1):
        cols[f'sin_{k}'] = np.sin(2 * np.pi * k * t)
        cols[f'cos_{k}'] = np.cos(2 * np.pi * k * t)
    cols['trend'] = t
    return pd.DataFrame(cols, index=index)

from sklearn.linear_model import Ridge

train_fourier = make_fourier_features(ts_train.index)
full_fourier  = make_fourier_features(ts_4h.index, n_terms=6)

# Use larger window for Prophet-style
train_f = full_fourier.loc[ts_train.index]
test_f  = full_fourier.loc[ts_test.index]

prophet_model = Ridge(alpha=1.0)
prophet_model.fit(train_f.values, ts_train.values)
prophet_pred  = prophet_model.predict(test_f.values)
prophet_mae   = mean_absolute_error(ts_test.values, prophet_pred)
prophet_rmse  = np.sqrt(mean_squared_error(ts_test.values, prophet_pred))
print(f"  Prophet-style  MAE={prophet_mae:.4f}  RMSE={prophet_rmse:.4f}")

# ─────────────────────────────────────────────
# 8. MAIN FIGURE
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(22, 18), facecolor='#0f1117')
fig.suptitle('Task 3: Household Energy Consumption — Time Series Forecasting',
             fontsize=17, fontweight='bold', color='white', y=0.99)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.30)

model_results = {
    'ARIMA(12,1,0)\n(Manual AR)':        {'pred': arima_pred,  'mae': arima_mae,    'rmse': arima_rmse,    'color':'#4FC3F7'},
    'GradientBoosting\n(XGBoost-style)': {'pred': gbr_pred,    'mae': gbr_mae,      'rmse': gbr_rmse,      'color':'#81C784'},
    'Prophet-Style\n(Fourier+Trend)':    {'pred': prophet_pred,'mae': prophet_mae,  'rmse': prophet_rmse,  'color':'#FFB74D'},
}

# ── Row 0: Full series overview ──
ax_full = fig.add_subplot(gs[0, :])
ax_full.set_facecolor('#1a1d27')
ax_full.plot(ts_4h.index, ts_4h.values, color='#aaa', lw=0.8, alpha=0.6, label='Historical')
ax_full.plot(ts_train.index, ts_train.values, color='#64B5F6', lw=1.0, label='Train')
ax_full.plot(ts_test.index, ts_test.values, color='white', lw=1.5, label='Actual (Test)')
ax_full.axvline(ts_test.index[0], color='#FFB74D', ls='--', lw=1.5, label='Forecast Start')
for name, res in model_results.items():
    ax_full.plot(ts_test.index, res['pred'], color=res['color'], lw=2,
                 ls='--', label=f"{name.split(chr(10))[0]} (RMSE={res['rmse']:.3f})")
ax_full.set_title('Full Time Series with Forecasts (4-Hourly, 60 Days)', color='white', fontsize=12)
ax_full.set_xlabel('Date', color='white')
ax_full.set_ylabel('Active Power (kW)', color='white')
ax_full.legend(facecolor='#252836', labelcolor='white', fontsize=9, ncol=3)
ax_full.tick_params(colors='white')
for sp in ax_full.spines.values(): sp.set_edgecolor('#333')
ax_full.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax_full.grid(alpha=0.10, color='white')

# ── Row 1: Per-model actual vs forecast (zoom on test) ──
model_items = list(model_results.items())
for col_i, (name, res) in enumerate(model_items[:2]):
    ax = fig.add_subplot(gs[1, col_i])
    ax.set_facecolor('#1a1d27')
    ax.plot(ts_test.index, ts_test.values, color='white', lw=2, label='Actual')
    ax.plot(ts_test.index, res['pred'], color=res['color'], lw=2, ls='--', label='Forecast')
    ax.fill_between(ts_test.index, ts_test.values, res['pred'],
                    alpha=0.15, color=res['color'])
    ax.set_title(f"{name}\nMAE={res['mae']:.4f}  RMSE={res['rmse']:.4f}",
                 color='white', fontsize=10)
    ax.set_xlabel('Date', color='white')
    ax.set_ylabel('Power (kW)', color='white')
    ax.legend(facecolor='#252836', labelcolor='white', fontsize=9)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.grid(alpha=0.10, color='white')

# ── Row 2: Third model + Metrics comparison ──
ax_third = fig.add_subplot(gs[2, 0])
ax_third.set_facecolor('#1a1d27')
name3, res3 = model_items[2]
ax_third.plot(ts_test.index, ts_test.values, color='white', lw=2, label='Actual')
ax_third.plot(ts_test.index, res3['pred'], color=res3['color'], lw=2, ls='--', label='Forecast')
ax_third.fill_between(ts_test.index, ts_test.values, res3['pred'],
                      alpha=0.15, color=res3['color'])
ax_third.set_title(f"{name3}\nMAE={res3['mae']:.4f}  RMSE={res3['rmse']:.4f}",
                   color='white', fontsize=10)
ax_third.set_xlabel('Date', color='white')
ax_third.set_ylabel('Power (kW)', color='white')
ax_third.legend(facecolor='#252836', labelcolor='white', fontsize=9)
ax_third.tick_params(colors='white')
for sp in ax_third.spines.values(): sp.set_edgecolor('#333')
ax_third.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax_third.grid(alpha=0.10, color='white')

# ── Metrics Comparison Bar Chart ──
ax_metrics = fig.add_subplot(gs[2, 1])
ax_metrics.set_facecolor('#1a1d27')
model_names_short = ['ARIMA', 'GBR', 'Prophet']
maes  = [r['mae']  for r in model_results.values()]
rmses = [r['rmse'] for r in model_results.values()]
x = np.arange(3)
b1 = ax_metrics.bar(x - 0.2, maes,  0.35, label='MAE',  color='#4FC3F7', alpha=0.85)
b2 = ax_metrics.bar(x + 0.2, rmses, 0.35, label='RMSE', color='#FFB74D', alpha=0.85)
for bar in list(b1) + list(b2):
    ax_metrics.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{bar.get_height():.3f}', ha='center', va='bottom',
                    color='white', fontsize=8)
ax_metrics.set_xticks(x)
ax_metrics.set_xticklabels(model_names_short, color='white', fontsize=10)
ax_metrics.set_title('Model Comparison: MAE & RMSE', color='white', fontsize=11)
ax_metrics.set_ylabel('Error (kW)', color='white')
ax_metrics.legend(facecolor='#252836', labelcolor='white')
ax_metrics.tick_params(colors='white')
for sp in ax_metrics.spines.values(): sp.set_edgecolor('#333')
ax_metrics.grid(axis='y', alpha=0.12, color='white')

# Feature importance for GBR
ax_fi_inset = ax_metrics.inset_axes([0.0, -0.75, 1.0, 0.6])
ax_fi_inset.set_facecolor('#1a1d27')
fi   = gbr.feature_importances_
fi_idx = np.argsort(fi)[-10:]
ax_fi_inset.barh(range(len(fi_idx)), fi[fi_idx], color='#81C784', alpha=0.8)
ax_fi_inset.set_yticks(range(len(fi_idx)))
ax_fi_inset.set_yticklabels([feat_cols[i] for i in fi_idx], color='white', fontsize=7)
ax_fi_inset.set_title('GBR Top Features', color='white', fontsize=9, pad=4)
ax_fi_inset.tick_params(colors='white', labelsize=7)
for sp in ax_fi_inset.spines.values(): sp.set_edgecolor('#333')
ax_fi_inset.set_facecolor('#1a1d27')

plt.savefig('./task3_forecasting.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("Saved: task3_forecasting.png")
print("\n Task 3 complete.")

Time series shape: (1440,)
Date range: 2007-01-01 00:00:00 → 2007-03-01 23:00:00
Stats: mean=1.257  std=0.506  min=0.166  max=2.679

Resampled (4h) shape: (360,)
Train: 318  |  Test: 42

Feature matrix: 23 features

Variance ratio (diff/orig): 1.3410
  → Likely Stationary 

Fitting ARIMA-style model (AR lag regression with differencing)...
  ARIMA(12,1,0)  MAE=0.0755  RMSE=0.0939
Fitting GradientBoostingRegressor (XGBoost-style)...
  GBR     MAE=0.0602  RMSE=0.0833
Fitting Prophet-style (Fourier seasonality) model...
  Prophet-style  MAE=0.4744  RMSE=0.5634
